# 🍎 Apple Game RL Training

| 단계 | 위치 | 내용 |
|------|------|------|
| **Phase 1** | 로컬 PC (Go) | `sl_data.bin` 생성 |
| **Phase 2** | 이 노트북 | SL 학습 → `model_sl.onnx` |
| **Phase 3** | 이 노트북 | PPO RL → `model_rl.onnx` |
| **Phase 4** | 로컬 PC (Go) | ModelAnA vs AnA 비교 |

### 사전 준비
1. 로컬에서 데이터 생성 (약 85초):
   ```bash
   go run . -gen-sl 500 -sl-out sl_data.bin
   ```
2. `sl_data.bin` (~228 MB) 을 Google Drive → `내 드라이브/apple_game/` 폴더에 업로드
3. **런타임 → 런타임 유형 변경 → T4 GPU 선택**
4. 아래 `GITHUB_URL` 수정 후 위에서부터 순서대로 실행

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  설정  —  여기만 수정하면 됩니다                              ║
# ╚══════════════════════════════════════════════════════════════╝

GITHUB_URL   = 'https://github.com/scheveningen361/10_apple_game.git'  # ← 본인 repo
DRIVE_FOLDER = '/content/drive/MyDrive/apple_game'             # Drive 저장 경로

# SL 설정
SL_EPOCHS = 50
SL_BATCH  = 2048

# RL 설정
RL_ITERS    = 200
RL_EPISODES = 200

In [ ]:
# ── 1. GPU 확인 ─────────────────────────────────────────────────────────────────
import torch, os

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {name}  ({vram:.1f} GB VRAM)')
else:
    print('⚠️  GPU 없음 — 런타임 유형을 T4로 변경하세요 (런타임 → 런타임 유형 변경)')

In [ ]:
# ── 2. Google Drive 마운트 ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_FOLDER, exist_ok=True)
print('Drive 마운트 완료:', DRIVE_FOLDER)

In [ ]:
# ── 3. GitHub repo 클론 / 업데이트 ─────────────────────────────────────────────
REPO_DIR = '/content/apple_game'
RL_DIR   = REPO_DIR + '/rl'

if os.path.exists(REPO_DIR + '/.git'):
    print('Repo 이미 존재 — git pull')
    !cd {REPO_DIR} && git pull
else:
    print('Repo 클론 중...')
    !git clone {GITHUB_URL} {REPO_DIR}

os.chdir(RL_DIR)
print('\n작업 디렉토리:', os.getcwd())
!ls -lh

In [ ]:
# ── 4. sl_data.bin Drive → Colab 복사 ──────────────────────────────────────────
import shutil

src = f'{DRIVE_FOLDER}/sl_data.bin'
dst = f'{RL_DIR}/sl_data.bin'

if os.path.exists(dst):
    mb = os.path.getsize(dst) / 1e6
    print(f'✅ sl_data.bin 이미 있음 ({mb:.0f} MB)')
elif os.path.exists(src):
    print(f'Drive에서 복사 중...')
    shutil.copy(src, dst)
    mb = os.path.getsize(dst) / 1e6
    print(f'✅ 복사 완료 ({mb:.0f} MB)')
else:
    print('❌ sl_data.bin 없음!')
    print(f'로컬 PC에서 생성 후 Drive의 {DRIVE_FOLDER}/ 폴더에 업로드하세요:')
    print('   go run . -gen-sl 500 -sl-out sl_data.bin')
    raise FileNotFoundError('sl_data.bin not found')

---
## Phase 2: SL 학습

- **모델**: ResNet (6 blocks, 128ch) + 보조 특징 3개 → **~1.8M params**
- **라벨**: `playGreedy(b2)` (0–170)
- **목표**: val RMSE < 3.0
- **예상 시간**: T4 기준 약 20–30분 (50 epoch)

In [ ]:
# ── Phase 2: SL 학습 ────────────────────────────────────────────────────────────
!python train_sl.py \
    --data    sl_data.bin \
    --out     model_sl.pt \
    --epochs  {SL_EPOCHS} \
    --batch   {SL_BATCH}  \
    --workers 2

In [ ]:
# ── SL 체크포인트 확인 ──────────────────────────────────────────────────────────
import torch
ckpt = torch.load('model_sl.pt', map_location='cpu')
print(f"Best val RMSE : {ckpt['best_val'] * 170:.4f}")
print(f"Epoch         : {ckpt['epoch'] + 1}")
print(f"Config        : {ckpt['config']}")

In [ ]:
# ── SL → ONNX 변환 ─────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, RL_DIR)
from train_sl import AppleNetSL, export_onnx

ckpt  = torch.load('model_sl.pt', map_location='cpu')
cfg   = ckpt['config']
model = AppleNetSL(channels=cfg['channels'], n_blocks=cfg['blocks'])
model.load_state_dict(ckpt['model'])

export_onnx(model, 'model_sl.onnx', device='cpu')
print(f"model_sl.onnx  {os.path.getsize('model_sl.onnx')/1e6:.1f} MB")

In [ ]:
# ── SL 모델 Drive 백업 ─────────────────────────────────────────────────────────
for f in ['model_sl.pt', 'model_sl.onnx']:
    if os.path.exists(f):
        shutil.copy(f, f'{DRIVE_FOLDER}/{f}')
        print(f'💾 {DRIVE_FOLDER}/{f}')

---
## Phase 3: PPO RL 학습

- **시작점**: SL 모델 (`model_sl.pt`)
- **방법**: Python 게임 엔진으로 자기 플레이 → 실제 점수로 라벨 업데이트
- **목표**: 평균 점수 > AnA (131.6점)
- **예상 시간**: T4 기준 약 1–2시간 (200 iter)

In [ ]:
# ── Phase 3: PPO RL 학습 ────────────────────────────────────────────────────────
!python train_rl.py \
    --sl       model_sl.pt \
    --out      model_rl.pt \
    --iters    {RL_ITERS}    \
    --episodes {RL_EPISODES}

In [ ]:
# ── RL 체크포인트 확인 ──────────────────────────────────────────────────────────
ckpt_rl = torch.load('model_rl.pt', map_location='cpu')
print(f"Best mean score : {ckpt_rl['best_score']:.2f}")
print(f"Iter            : {ckpt_rl['iter'] + 1}")
print(f"AnA baseline    : ~131.6  →  gain {ckpt_rl['best_score'] - 131.6:+.2f}")

In [ ]:
# ── RL → ONNX 변환 ─────────────────────────────────────────────────────────────
from train_sl import AppleNetSL, export_onnx

ckpt_rl = torch.load('model_rl.pt', map_location='cpu')
cfg     = ckpt_rl['config']
model_rl = AppleNetSL(channels=cfg['channels'], n_blocks=cfg['blocks'])
model_rl.load_state_dict(ckpt_rl['model'])

export_onnx(model_rl, 'model_rl.onnx', device='cpu')
print(f"model_rl.onnx  {os.path.getsize('model_rl.onnx')/1e6:.1f} MB")

In [ ]:
# ── RL 모델 Drive 백업 + 최종 요약 ────────────────────────────────────────────
for f in ['model_rl.pt', 'model_rl.onnx']:
    if os.path.exists(f):
        shutil.copy(f, f'{DRIVE_FOLDER}/{f}')
        print(f'💾 {DRIVE_FOLDER}/{f}')

print('\n' + '='*50)
print('학습 완료!')
print(f'Drive 경로: {DRIVE_FOLDER}')
!ls -lh {DRIVE_FOLDER}

---
## Phase 4: 로컬 평가

Drive에서 `model_rl.onnx` 다운로드 후 로컬 PC에서:

```bash
go run -tags nn . -nn-ana -n 100 -model model_rl.onnx -ort-lib onnxruntime.dll
```

| 알고리즘 | 목표 점수 |
|----------|----------|
| AnA | ~131.6 |
| ModelAnA (SL) | ~133–135 |
| ModelAnA (RL) | ~135–140 |